# ЛР №3. Профилирование и качество данных: raw → staging

**Датасет:** `hour_data` (UCI Bike Sharing Dataset, почасовые данные Capital Bikeshare).
**Входной S3-путь:** `s3://raw/hour_data/ingested_on=2026-09-16/hour.csv`

**Аналитический вопрос:** Как погодные условия и календарные факторы влияют на суточное и почасовое количество аренд велосипедов? Различается ли поведение зарегистрированных и случайных пользователей?

**Смысл одной строки:** один час наблюдения: дата, час, календарные и погодные признаки, количество аренд.

**Бизнес-ключ:** составной `dteday + hr`.

In [1]:
DATASET_SLUG = "hour_data"
INGESTED_ON = "2026-09-16"

INPUT_S3_PATH = f"s3://raw/{DATASET_SLUG}/ingested_on={INGESTED_ON}/hour.csv"
OUTPUT_PREFIX = f"s3://staging/{DATASET_SLUG}/source_ingested_on={INGESTED_ON}/"
STAGING_S3_PATH = OUTPUT_PREFIX + "staging.parquet"
QUARANTINE_S3_PATH = OUTPUT_PREFIX + "quarantine.parquet"

MAX_DUPLICATE_KEY_ROWS = 0
MAX_CNT_NULL_ROWS = 0
MAX_HR_OUT_OF_RANGE_ROWS = 0
MAX_WEATHERSIT_OUT_OF_RANGE_ROWS = 0
MAX_CONSISTENCY_VIOLATIONS = 0

print("INPUT_S3_PATH      =", INPUT_S3_PATH)
print("STAGING_S3_PATH    =", STAGING_S3_PATH)
print("QUARANTINE_S3_PATH =", QUARANTINE_S3_PATH)

INPUT_S3_PATH      = s3://raw/hour_data/ingested_on=2026-09-16/hour.csv
STAGING_S3_PATH    = s3://staging/hour_data/source_ingested_on=2026-09-16/staging.parquet
QUARANTINE_S3_PATH = s3://staging/hour_data/source_ingested_on=2026-09-16/quarantine.parquet


In [2]:
import os
import duckdb
from IPython.display import display

def get_s3_endpoint() -> str:
    endpoint = os.getenv("S3_ENDPOINT", "seaweedfs:8333")
    endpoint = endpoint.replace("http://", "").replace("https://", "").rstrip("/")
    return endpoint

con = duckdb.connect()

try:
    con.execute("INSTALL httpfs")
except Exception:
    pass
con.execute("LOAD httpfs")

con.execute(f"SET s3_endpoint = '{get_s3_endpoint()}'")
con.execute("SET s3_url_style = 'path'")
con.execute("SET s3_use_ssl = false")
con.execute("SET s3_region = 'us-east-1'")
con.execute(f"SET s3_access_key_id = '{os.getenv('S3_ACCESS_KEY', 's3admin')}'")
con.execute(f"SET s3_secret_access_key = '{os.getenv('S3_SECRET_KEY', 's3admin123')}'")

con.execute(
    "CREATE OR REPLACE TEMP VIEW raw_csv AS "
    f"SELECT * FROM read_csv_auto('{INPUT_S3_PATH}', header=true)"
)

display(con.execute("SELECT * FROM raw_csv LIMIT 5").fetchdf())

rule_results = []

def run_rule(rule_id, title, sql, params, threshold, action):
    checked, violations = con.execute(sql, params).fetchone()
    checked = int(checked or 0)
    violations = int(violations or 0)
    share = violations / checked if checked else 0.0
    passed = violations <= threshold
    print(f"{rule_id} | {title}")
    print(f"  проверено строк: {checked}")
    print(f"  нарушений: {violations} ({share:.4%})")
    print(f"  порог: {threshold}; результат: {'PASS' if passed else 'FAIL'}; действие: {action}")
    return {
        "rule_id": rule_id,
        "title": title,
        "checked": checked,
        "violations": violations,
        "share": share,
        "threshold": threshold,
        "passed": passed,
        "action": action,
    }

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,ingested_on
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2026-09-16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2026-09-16
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2026-09-16
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13,2026-09-16
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1,2026-09-16


## Профиль выбранных полей
Профилируются поля `dteday`, `hr`, `cnt`, `weathersit`. Пропуски недопустимы для этих полей.

In [3]:
display(con.execute("SELECT COUNT(*) AS row_count FROM raw_csv").fetchdf())
display(con.execute("DESCRIBE SELECT * FROM raw_csv").fetchdf())
display(con.execute("SELECT * FROM raw_csv LIMIT 3").fetchdf())

display(con.execute("""
SELECT 'dteday' AS field, COUNT(*) AS checked_rows,
       SUM(CASE WHEN dteday IS NULL THEN 1 ELSE 0 END) AS null_rows
FROM raw_csv
UNION ALL
SELECT 'hr', COUNT(*), SUM(CASE WHEN hr IS NULL THEN 1 ELSE 0 END) FROM raw_csv
UNION ALL
SELECT 'cnt', COUNT(*), SUM(CASE WHEN cnt IS NULL THEN 1 ELSE 0 END) FROM raw_csv
UNION ALL
SELECT 'weathersit', COUNT(*), SUM(CASE WHEN weathersit IS NULL THEN 1 ELSE 0 END) FROM raw_csv
""").fetchdf())

display(con.execute("""
WITH src AS (SELECT * FROM raw_csv),
dup AS (SELECT dteday, hr, COUNT(*) AS n FROM src GROUP BY dteday, hr HAVING COUNT(*) > 1)
SELECT COALESCE(SUM(n), 0) AS duplicate_key_rows, COUNT(*) AS duplicate_key_groups FROM dup
""").fetchdf())

display(con.execute("SELECT MIN(dteday) AS min_event_date, MAX(dteday) AS max_event_date FROM raw_csv").fetchdf())

display(con.execute("""
SELECT hr, COUNT(*) AS rows_cnt, SUM(cnt) AS total_cnt, AVG(cnt) AS avg_cnt
FROM raw_csv GROUP BY hr ORDER BY hr
""").fetchdf())

,row_count
0,17379


,column_name,column_type,null,key,default,extra
0,instant,BIGINT,YES,None,None,None
1,dteday,DATE,YES,None,None,None
2,season,BIGINT,YES,None,None,None
3,yr,BIGINT,YES,None,None,None
4,mnth,BIGINT,YES,None,None,None
5,hr,BIGINT,YES,None,None,None
6,holiday,BIGINT,YES,None,None,None
7,weekday,BIGINT,YES,None,None,None
8,workingday,BIGINT,YES,None,None,None
9,weathersit,BIGINT,YES,None,None,None


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,ingested_on
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16,2026-09-16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40,2026-09-16
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32,2026-09-16


,field,checked_rows,null_rows
0,dteday,17379,0.0
1,hr,17379,0.0
2,cnt,17379,0.0
3,weathersit,17379,0.0


,duplicate_key_rows,duplicate_key_groups
0,0.0,0


,min_event_date,max_event_date
0,2011-01-01,2012-12-31


,hr,rows_cnt,total_cnt,avg_cnt
0,0,726,39130.0,53.898072
1,1,724,24164.0,33.375691
2,2,715,16352.0,22.869930
3,3,697,8174.0,11.727403
4,4,697,4428.0,6.352941
5,5,717,14261.0,19.889819
6,6,725,55132.0,76.044138
7,7,727,154171.0,212.064649
8,8,727,261001.0,359.011004
9,9,727,159438.0,219.309491


### Правило R1_UNIQUE_KEY
- **Поле/ключ:** `dteday + hr`.
- **Проверяемые строки:** все строки raw.
- **Условие:** нет повторяющихся пар `dteday + hr`.
- **Порог:** `MAX_DUPLICATE_KEY_ROWS = 0`.
- **Критичность:** critical.
- **Действие:** quarantine.
- **Связь с метрикой:** дубли искажают почасовой ряд.

In [4]:
rule_results.append(run_rule(
    "R1_UNIQUE_KEY", "Уникальность бизнес-ключа dteday + hr",
    """
    WITH src AS (SELECT * FROM raw_csv),
    dup_keys AS (SELECT dteday, hr, 1 AS _dup FROM src GROUP BY dteday, hr HAVING COUNT(*) > 1)
    SELECT (SELECT COUNT(*) FROM src) AS checked_rows,
           (SELECT COUNT(*) FROM src s JOIN dup_keys d
            ON d.dteday IS NOT DISTINCT FROM s.dteday AND d.hr IS NOT DISTINCT FROM s.hr) AS violations
    """, [], MAX_DUPLICATE_KEY_ROWS, "critical / quarantine"))

R1_UNIQUE_KEY | Уникальность бизнес-ключа dteday + hr
  проверено строк: 17379
  нарушений: 0 (0.0000%)
  порог: 0; результат: PASS; действие: critical / quarantine


### Правило R2_CNT_NOT_NULL
- **Поле:** `cnt`.
- **Проверяемые строки:** все строки raw.
- **Условие:** `cnt IS NOT NULL`.
- **Порог:** `MAX_CNT_NULL_ROWS = 0`.
- **Критичность:** critical.
- **Действие:** quarantine.
- **Связь с метрикой:** `cnt` — основная метрика спроса.

In [5]:
rule_results.append(run_rule(
    "R2_CNT_NOT_NULL", "Полнота cnt",
    """
    SELECT COUNT(*) AS checked_rows,
           COALESCE(SUM(CASE WHEN cnt IS NULL THEN 1 ELSE 0 END), 0) AS violations
    FROM raw_csv
    """, [], MAX_CNT_NULL_ROWS, "critical / quarantine"))

R2_CNT_NOT_NULL | Полнота cnt
  проверено строк: 17379
  нарушений: 0 (0.0000%)
  порог: 0; результат: PASS; действие: critical / quarantine


### Правило R3_HR_RANGE
- **Поле:** `hr`.
- **Проверяемые строки:** все строки raw.
- **Условие:** `hr BETWEEN 0 AND 23`.
- **Порог:** `MAX_HR_OUT_OF_RANGE_ROWS = 0`.
- **Критичность:** critical.
- **Действие:** quarantine.
- **Связь с метрикой:** час суток для внутрисуточной динамики.

In [6]:
rule_results.append(run_rule(
    "R3_HR_RANGE", "Допустимость hr: 0..23",
    """
    SELECT COUNT(*) AS checked_rows,
           COALESCE(SUM(CASE WHEN hr IS NULL OR hr < 0 OR hr > 23 THEN 1 ELSE 0 END), 0) AS violations
    FROM raw_csv
    """, [], MAX_HR_OUT_OF_RANGE_ROWS, "critical / quarantine"))

R3_HR_RANGE | Допустимость hr: 0..23
  проверено строк: 17379
  нарушений: 0 (0.0000%)
  порог: 0; результат: PASS; действие: critical / quarantine


### Правило R4_WEATHERSIT_RANGE
- **Поле:** `weathersit`.
- **Проверяемые строки:** все строки raw.
- **Условие:** `weathersit BETWEEN 1 AND 4`.
- **Порог:** `MAX_WEATHERSIT_OUT_OF_RANGE_ROWS = 0`.
- **Критичность:** warning.
- **Действие:** flag.
- **Связь с метрикой:** тип погоды как срез.

In [7]:
rule_results.append(run_rule(
    "R4_WEATHERSIT_RANGE", "Допустимость weathersit: 1..4",
    """
    SELECT COUNT(*) AS checked_rows,
           COALESCE(SUM(CASE WHEN weathersit IS NULL OR weathersit NOT BETWEEN 1 AND 4 THEN 1 ELSE 0 END), 0) AS violations
    FROM raw_csv
    """, [], MAX_WEATHERSIT_OUT_OF_RANGE_ROWS, "warning / flag"))

R4_WEATHERSIT_RANGE | Допустимость weathersit: 1..4
  проверено строк: 17379
  нарушений: 0 (0.0000%)
  порог: 0; результат: PASS; действие: warning / flag


### Правило R5_CASUAL_REGISTERED_CNT
- **Поля:** `casual`, `registered`, `cnt`.
- **Проверяемые строки:** все строки raw.
- **Условие:** `casual + registered = cnt`.
- **Порог:** `MAX_CONSISTENCY_VIOLATIONS = 0`.
- **Критичность:** critical.
- **Действие:** quarantine.
- **Связь с метрикой:** согласованность сегментов и общей метрики.

In [8]:
rule_results.append(run_rule(
    "R5_CASUAL_REGISTERED_CNT", "Согласованность casual + registered = cnt",
    """
    SELECT COUNT(*) AS checked_rows,
           COALESCE(SUM(CASE WHEN (casual + registered) IS DISTINCT FROM cnt THEN 1 ELSE 0 END), 0) AS violations
    FROM raw_csv
    """, [], MAX_CONSISTENCY_VIOLATIONS, "critical / quarantine"))

R5_CASUAL_REGISTERED_CNT | Согласованность casual + registered = cnt
  проверено строк: 17379
  нарушений: 0 (0.0000%)
  порог: 0; результат: PASS; действие: critical / quarantine


In [9]:
con.execute("DROP TABLE IF EXISTS src_rows")
con.execute("""
CREATE TEMP TABLE src_rows AS
SELECT row_number() OVER () AS _row_id,
       CAST(instant AS BIGINT) AS instant, CAST(dteday AS DATE) AS dteday,
       CAST(season AS INTEGER) AS season, CAST(yr AS INTEGER) AS yr,
       CAST(mnth AS INTEGER) AS mnth, CAST(hr AS INTEGER) AS hr,
       CAST(holiday AS INTEGER) AS holiday, CAST(weekday AS INTEGER) AS weekday,
       CAST(workingday AS INTEGER) AS workingday, CAST(weathersit AS INTEGER) AS weathersit,
       CAST(temp AS DOUBLE) AS temp, CAST(atemp AS DOUBLE) AS atemp,
       CAST(hum AS DOUBLE) AS hum, CAST(windspeed AS DOUBLE) AS windspeed,
       CAST(casual AS BIGINT) AS casual, CAST(registered AS BIGINT) AS registered,
       CAST(cnt AS BIGINT) AS cnt
FROM raw_csv
""")

con.execute("DROP TABLE IF EXISTS rule_flags")
con.execute("""
CREATE TEMP TABLE rule_flags AS
WITH dup_keys AS (SELECT dteday, hr, 1 AS _dup FROM src_rows GROUP BY dteday, hr HAVING COUNT(*) > 1)
SELECT s._row_id,
       COALESCE(d._dup, 0) AS r1_dup,
       CASE WHEN s.cnt IS NULL THEN 1 ELSE 0 END AS r2_cnt_null,
       CASE WHEN s.hr IS NULL OR s.hr < 0 OR s.hr > 23 THEN 1 ELSE 0 END AS r3_hr_bad,
       CASE WHEN s.weathersit IS NULL OR s.weathersit NOT BETWEEN 1 AND 4 THEN 1 ELSE 0 END AS r4_weather_bad,
       CASE WHEN (s.casual + s.registered) IS DISTINCT FROM s.cnt THEN 1 ELSE 0 END AS r5_consistency_bad
FROM src_rows s
LEFT JOIN dup_keys d ON d.dteday IS NOT DISTINCT FROM s.dteday AND d.hr IS NOT DISTINCT FROM s.hr
""")

con.execute("DROP TABLE IF EXISTS violations")
con.execute("""
CREATE TEMP TABLE violations AS
SELECT _row_id, 'R1_UNIQUE_KEY' AS quality_rule_id, 'duplicate business key (dteday+hr)' AS quarantine_reason FROM rule_flags WHERE r1_dup = 1
UNION ALL SELECT _row_id, 'R2_CNT_NOT_NULL', 'cnt is NULL' FROM rule_flags WHERE r2_cnt_null = 1
UNION ALL SELECT _row_id, 'R3_HR_RANGE', 'hr outside 0..23' FROM rule_flags WHERE r3_hr_bad = 1
UNION ALL SELECT _row_id, 'R4_WEATHERSIT_RANGE', 'weathersit outside 1..4' FROM rule_flags WHERE r4_weather_bad = 1
UNION ALL SELECT _row_id, 'R5_CASUAL_REGISTERED_CNT', 'casual + registered != cnt' FROM rule_flags WHERE r5_consistency_bad = 1
""")

con.execute("DROP TABLE IF EXISTS critical_agg")
con.execute("""
CREATE TEMP TABLE critical_agg AS
SELECT _row_id, string_agg(quality_rule_id, ', ' ORDER BY quality_rule_id) AS critical_rule_ids,
       string_agg(quarantine_reason, '; ' ORDER BY quality_rule_id) AS critical_reason
FROM violations
WHERE quality_rule_id IN ('R1_UNIQUE_KEY','R2_CNT_NOT_NULL','R3_HR_RANGE','R5_CASUAL_REGISTERED_CNT')
GROUP BY _row_id
""")

con.execute("DROP TABLE IF EXISTS warning_agg")
con.execute("""
CREATE TEMP TABLE warning_agg AS
SELECT _row_id, string_agg(quality_rule_id, ', ' ORDER BY quality_rule_id) AS warning_rule_ids
FROM violations WHERE quality_rule_id = 'R4_WEATHERSIT_RANGE' GROUP BY _row_id
""")

con.execute("DROP TABLE IF EXISTS violation_agg")
con.execute("""
CREATE TEMP TABLE violation_agg AS
SELECT _row_id, string_agg(quality_rule_id, ', ' ORDER BY quality_rule_id) AS quality_rule_ids,
       string_agg(quarantine_reason, '; ' ORDER BY quality_rule_id) AS quarantine_reason
FROM violations GROUP BY _row_id
""")

con.execute("DROP TABLE IF EXISTS staging")
con.execute("""
CREATE TEMP TABLE staging AS
SELECT s.instant, s.dteday, s.season, s.yr, s.mnth, s.hr, s.holiday, s.weekday, s.workingday,
       s.weathersit, s.temp, s.atemp, s.hum, s.windspeed, s.casual, s.registered, s.cnt,
       CASE WHEN w._row_id IS NULL THEN 'clean' ELSE 'flagged' END AS quality_status,
       CASE WHEN w._row_id IS NULL THEN 0 ELSE 1 END AS quality_flag,
       COALESCE(w.warning_rule_ids, '') AS quality_rule_ids
FROM src_rows s
LEFT JOIN critical_agg c USING (_row_id)
LEFT JOIN warning_agg w USING (_row_id)
WHERE c._row_id IS NULL
""")

con.execute("DROP TABLE IF EXISTS quarantine")
con.execute("""
CREATE TEMP TABLE quarantine AS
SELECT s.instant, s.dteday, s.season, s.yr, s.mnth, s.hr, s.holiday, s.weekday, s.workingday,
       s.weathersit, s.temp, s.atemp, s.hum, s.windspeed, s.casual, s.registered, s.cnt,
       v.quality_rule_ids, v.quarantine_reason
FROM src_rows s
JOIN critical_agg c USING (_row_id)
JOIN violation_agg v USING (_row_id)
""")

con.execute(f"COPY (SELECT * FROM staging) TO '{STAGING_S3_PATH}' (FORMAT PARQUET);")
con.execute(f"COPY (SELECT * FROM quarantine) TO '{QUARANTINE_S3_PATH}' (FORMAT PARQUET);")

raw_count = con.execute("SELECT COUNT(*) FROM src_rows").fetchone()[0]
staging_count = con.execute("SELECT COUNT(*) FROM staging").fetchone()[0]
quarantine_count = con.execute("SELECT COUNT(*) FROM quarantine").fetchone()[0]

print("raw rows       =", raw_count)
print("staging rows   =", staging_count)
print("quarantine rows=", quarantine_count)
assert raw_count == staging_count + quarantine_count, "Нарушен баланс raw = staging + quarantine"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw rows       = 17379
staging rows   = 17379
quarantine rows= 0


In [10]:
staging_s3_count = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [STAGING_S3_PATH]).fetchone()[0]
quarantine_s3_count = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [QUARANTINE_S3_PATH]).fetchone()[0]

print("staging.parquet rows   =", staging_s3_count)
print("quarantine.parquet rows=", quarantine_s3_count)
print("raw = staging + quarantine:", raw_count == staging_s3_count + quarantine_s3_count)
assert raw_count == staging_s3_count + quarantine_s3_count

# Проверка критических правил на staging
print("--- Проверка критических правил на staging ---")
print(con.execute("SELECT COUNT(*) FROM read_parquet(?) WHERE cnt IS NULL", [STAGING_S3_PATH]).fetchone()[0], "строк с cnt IS NULL")
print(con.execute("SELECT COUNT(*) FROM read_parquet(?) WHERE hr < 0 OR hr > 23", [STAGING_S3_PATH]).fetchone()[0], "строк с hr вне 0..23")
print(con.execute("SELECT COUNT(*) FROM read_parquet(?) WHERE (casual + registered) != cnt", [STAGING_S3_PATH]).fetchone()[0], "строк с нарушением casual + registered = cnt")

print("--- Пример записей карантина (если есть) ---")
display(con.execute("SELECT * FROM read_parquet(?) LIMIT 2", [QUARANTINE_S3_PATH]).fetchdf())
print("--- Пример flag-записей staging (если есть) ---")
display(con.execute("SELECT * FROM read_parquet(?) WHERE quality_flag = 1 LIMIT 2", [STAGING_S3_PATH]).fetchdf())

staging.parquet rows   = 17379
quarantine.parquet rows= 0
raw = staging + quarantine: True
--- Проверка критических правил на staging ---


0 строк с cnt IS NULL


0 строк с hr вне 0..23


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

0 строк с нарушением casual + registered = cnt
--- Пример записей карантина (если есть) ---


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,quality_rule_ids,quarantine_reason,source_ingested_on


--- Пример flag-записей staging (если есть) ---


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,...,atemp,hum,windspeed,casual,registered,cnt,quality_status,quality_flag,quality_rule_ids,source_ingested_on
